# Exercise 3 — Monte Carlo Reinforcement Learning 🚗
### Complete, student-friendly, runnable solution

**Learning goals:** (1) first-visit Monte Carlo policy evaluation, (2) on-policy ε-greedy Monte Carlo control, and (3) a harder rectangular racetrack.

**Run locally:** `python -m pip install numpy matplotlib tqdm ipykernel` then open this notebook in VS Code or Jupyter and choose your Python kernel. **Run cells from top to bottom.** No external course files are required.

This solution implements the racetrack rules directly, so you can study each component. Random seeds aid reproducibility, but reinforcement learning results vary across runs. MC control finds an *improved* policy, not a mathematically guaranteed globally optimal one.

In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt
from collections import defaultdict
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print('Ready! NumPy:', np.__version__)

## 0. The environment and the Markov decision process

A state is **(row, column, vertical velocity, horizontal velocity)**. Nine actions change each velocity component by −1, 0 or +1. Each move costs **−1**, so finishing sooner earns a higher return. Collision **ends the current episode** (the next episode restarts at the start line). This is an explicit modeling choice: the course notebook's environment sets `truncated=True` on collision; stopping and restarting is consistent with the supplied training templates. Reaching the finish line ends the episode successfully. A step limit prevents endless episodes.

In the visualized course, `W` = wall, `o` = road, `-` = start, `+` = finish.

In [ ]:
def build_uturn_course(course_dim=(8, 10), inner_wall_dim=(2, 6)):
    h, w = course_dim
    ih, iw = inner_wall_dim
    top, bottom = h//2-ih//2, h//2+ih//2
    street = w//2-iw//2
    rows = []
    for y in range(h):
        if y < top:
            half = w//2-1
            row = 'W'*(half//2+1)+'W-'+'o'*(half-1+half//2)
        elif y < bottom:
            row = 'W'*street+'W'*iw+'o'*street
        else:
            row = 'W'*(half//2+1)+'W+'+'o'*(half-1+half//2)
        rows.append(row)
    return ['W'*(w+2)] + ['W'+r+'W' for r in rows] + ['W'*(w+2)]

def build_rect_course(course_dim=(8, 10), inner_wall_dim=(2, 6)):
    h, w = course_dim
    ih, iw = inner_wall_dim
    top, bottom = h//2-ih//2, h//2+ih//2
    street = w//2-iw//2
    rows=[]
    for y in range(h):
        if y < top:
            half=w//2-1
            row='o'*half+'+W-'+'o'*(half-1)
        elif y < bottom:
            row='o'*street+'W'*iw+'o'*street
        else:
            row='o'*w
        rows.append(row)
    return ['W'*(w+2)] + ['W'+r+'W' for r in rows] + ['W'*(w+2)]

ACTIONS = [(dy, dx) for dy in (-1, 0, 1) for dx in (-1, 0, 1)]

class RaceTrack:
    def __init__(self, course, max_speed=5, seed=42):
        assert len({len(row) for row in course}) == 1, 'Rows must have equal lengths'
        self.grid=np.array([list(row) for row in course])
        self.h, self.w=self.grid.shape
        self.max_speed=max_speed
        self.starts=[tuple(p) for p in np.argwhere(self.grid=='-')]
        assert self.starts and np.any(self.grid=='+'), 'Need start and finish lines'
        self.rng=np.random.default_rng(seed)
        self.reset()

    def reset(self):
        self.y,self.x=map(int,self.starts[self.rng.integers(len(self.starts))])
        self.vy=self.vx=0
        return self.state

    @property
    def state(self):
        return (self.y,self.x,self.vy,self.vx)

    def step(self, action):
        dy,dx=ACTIONS[int(action)]
        vy=int(np.clip(self.vy+dy,-self.max_speed,self.max_speed))
        vx=int(np.clip(self.vx+dx,-self.max_speed,self.max_speed))
        # Check cells ALONG motion, not just the destination.
        # 2*max_speed interpolated segments approximate the original course environment.
        ys=np.linspace(self.y,self.y+vy,2*self.max_speed+1).astype(int)
        xs=np.linspace(self.x,self.x+vx,2*self.max_speed+1).astype(int)
        for y,x in zip(ys[1:],xs[1:]):
            if not (0<=y<self.h and 0<=x<self.w) or self.grid[y,x]=='W':
                return self.state,-1.0,False,True  # crash / truncated
            if self.grid[y,x]=='+':
                self.y,self.x,self.vy,self.vx=int(y),int(x),vy,vx
                return self.state,-1.0,True,False
        self.y=int(self.y+vy); self.x=int(self.x+vx)
        self.vy,self.vx=vy,vx
        return self.state,-1.0,False,False

def show_course(env, title='Racetrack'):
    palette={'W':0,'o':1,'-':2,'+':3}
    grid=np.vectorize(palette.get)(env.grid)
    fig,ax=plt.subplots(figsize=(7,5))
    ax.imshow(grid,cmap=plt.get_cmap('viridis',4),vmin=0,vmax=3)
    ax.set_xticks(range(env.w));ax.set_yticks(range(env.h));ax.grid(alpha=.2)
    ax.set_title(title);plt.show()

uturn=RaceTrack(build_uturn_course())
print('9 acceleration choices:',ACTIONS)
print('Example initial state:',uturn.state)
show_course(uturn,'U-turn course')

## 1. First-visit Monte Carlo prediction

**Prediction** means evaluate a *fixed* policy, not improve it.

For episode rewards \(r_{t+1},\ldots,r_T\), the return from step \(t\) is

\[G_t = r_{t+1} + \gamma r_{t+2} + \cdots + \gamma^{T-t-1}r_T.\]

For **first-visit** MC, update \(V(s)\) only at the earliest occurrence of state \(s\) in each episode. We use an incremental mean: \(V(s)\leftarrow V(s)+(G-V(s))/N(s)\).

Below is an intuitive heuristic policy: travel right across the top, turn down on the right, then travel left toward the lower finish. It is used for **evaluation only**.

In [ ]:
def dummy_policy(state, env):
    y,x,vy,vx=state
    if y <= env.h//2 and x < env.w-3:
        target=(0,1)      # encourage movement right
    elif y <= env.h//2:
        target=(1,0)      # turn down
    else:
        target=(0,-1)     # then turn left
    # Preserve low speed to make a safer, interpretable baseline.
    ay=np.sign(target[0]-vy)
    ax=np.sign(target[1]-vx)
    return ACTIONS.index((int(ay),int(ax)))

def run_episode(env, choose_action, max_steps=200, gamma=1.0):
    state=env.reset()
    episode=[]
    success=False
    for _ in range(max_steps):
        action=int(choose_action(state))
        next_state,reward,terminated,truncated=env.step(action)
        episode.append((state,action,reward))
        state=next_state
        if terminated or truncated:
            success=bool(terminated)
            break
    return episode,success

def first_visit_prediction(env, policy_fn, episodes=1500, max_steps=200, gamma=1.0):
    values=defaultdict(float)
    counts=defaultdict(int)
    lengths=[]; successes=[]
    for _ in range(episodes):
        episode,ok=run_episode(env,lambda s:policy_fn(s,env),max_steps)
        lengths.append(len(episode));successes.append(ok)
        G=0.0; returns=[0.0]*len(episode)
        for t in reversed(range(len(episode))):
            G=episode[t][2]+gamma*G
            returns[t]=G
        seen=set()
        for (state,action,reward),G in zip(episode,returns):
            if state not in seen:
                seen.add(state)
                counts[state]+=1
                values[state]+=(G-values[state])/counts[state]
    return values,counts,np.array(lengths),np.array(successes)

V,N,lengths,successes=first_visit_prediction(uturn,dummy_policy)
print(f'Evaluated states: {len(V)} | Successful episodes: {successes.mean():.1%}')
print('Sample estimated state values:',list(V.items())[:5])

### Visualize the prediction

We average **observed** state values across velocities for each position. Missing positions remain uncolored instead of being mistaken for zero-value states. With step rewards of −1 and γ=1, a value closer to zero means fewer remaining steps *under the evaluated policy* (including crash/episode termination).

In [ ]:
def plot_position_values(env,V,title='Estimated value by position'):
    total=np.zeros((env.h,env.w)); count=np.zeros((env.h,env.w))
    for (y,x,vy,vx),value in V.items():
        total[y,x]+=value;count[y,x]+=1
    out=np.full((env.h,env.w),np.nan)
    np.divide(total,count,out=out,where=count>0)
    plt.figure(figsize=(8,5));plt.imshow(out,cmap='plasma');plt.colorbar(label='Mean estimated return')
    plt.title(title);plt.show()
plot_position_values(uturn,V)
plt.figure(figsize=(8,3));plt.plot(lengths,alpha=.35);plt.title('Episode length (fixed policy)');plt.xlabel('Episode');plt.ylabel('Steps');plt.show()

## 2. On-policy ε-greedy first-visit Monte Carlo control

**Control** improves behavior by learning action values \(Q(s,a)\). At each step, the agent follows an **ε-greedy behavior policy**: with probability ε pick a uniformly random action; otherwise choose a greedy action using \(Q\). For every episode, update each distinct \((s,a)\) **only once**, using its first-visit return.

An important detail: since all step rewards are negative, initialize action values somewhat pessimistically (e.g. −200), rather than zero. Otherwise unseen actions can remain artificially attractive. We **warm-start** the first 250 training episodes with the supplied dummy policy; crashes/timeouts incur a −200 terminal training penalty so the agent does not learn to crash immediately to avoid step costs. Ties are broken randomly. To avoid episodes that last forever, `max_steps` caps episode length. A crash also ends an episode.

**Training tips:** More episodes typically produce better results, but Monte Carlo learning has high variance. Try ε between 0.05 and 0.2; compare performance using a greedy *evaluation* policy on fresh episodes.

In [ ]:
def epsilon_greedy(state,Q,rng,epsilon=0.1,initial_q=-200.0):
    if rng.random()<epsilon:
        return int(rng.integers(9))
    qs=np.array([Q.get((state,a),initial_q) for a in range(9)])
    best=np.flatnonzero(qs==qs.max())
    return int(rng.choice(best))

def mc_control(env,episodes=10000,epsilon=0.1,gamma=1.0,max_steps=200,
               initial_q=-200.0,seed=123,verbose=True):
    rng=np.random.default_rng(seed)
    Q={}; visits=defaultdict(int)
    lengths=[]; wins=[]
    for ep in range(episodes):
        episode,won=run_episode(env,(lambda s: dummy_policy(s,env)) if ep<250 else (lambda s:epsilon_greedy(s,Q,rng,epsilon,initial_q)),max_steps)
        G=0.0;returns=[0.0]*len(episode)
        for i in range(len(episode)-1,-1,-1):
            G=episode[i][2]+gamma*G
            if i==len(episode)-1 and not won: G-=200.0  # penalty for crash / timeout
            returns[i]=G
        seen=set()
        for (s,a,r),ret in zip(episode,returns):
            key=(s,a)
            if key in seen:continue
            seen.add(key)
            visits[key]+=1
            old=Q.get(key,initial_q)
            Q[key]=old+(ret-old)/visits[key]
        lengths.append(len(episode));wins.append(won)
        if verbose and (ep+1)%(max(episodes//5,1))==0:
            window=min(500,len(wins))
            print(f'{ep+1:>6} episodes | latest win rate: {np.mean(wins[-window:]):.1%} | avg steps: {np.mean(lengths[-window:]):.1f}')
    return Q,visits,np.array(lengths),np.array(wins)

# Running fewer episodes is fast; increase this to 30_000+ for a stronger policy.
TRAIN_EPISODES=10000
train_env=RaceTrack(build_uturn_course(),seed=SEED)
Q,SA_visits,train_lengths,train_wins=mc_control(train_env,episodes=TRAIN_EPISODES)
print('Distinct observed (state,action) pairs:',len(Q))

### Learning curve and held-out evaluation

Do **not** interpret training success rate alone as final policy quality: the training policy deliberately explores. Evaluation below always picks a greedy action (ε = 0) on independent episodes.

In [ ]:
def moving_average(x,window=250):
    if len(x)<window:return np.array([])
    return np.convolve(x,np.ones(window)/window,mode='valid')

fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].plot(moving_average(train_wins.astype(float)));axes[0].set_title('Smoothed training win rate')
axes[0].set_ylim(0,1);axes[0].set_xlabel('Episode');axes[0].set_ylabel('Win fraction')
axes[1].plot(moving_average(train_lengths));axes[1].set_title('Smoothed episode length')
axes[1].set_xlabel('Episode');axes[1].set_ylabel('Steps')
plt.tight_layout();plt.show()

def evaluate_greedy(env,Q,n=100,max_steps=200,seed=99,initial_q=-200.0):
    rng=np.random.default_rng(seed)
    lengths=[];wins=[];paths=[]
    for _ in range(n):
        episode,won=run_episode(env,lambda s:epsilon_greedy(s,Q,rng,0.,initial_q),max_steps)
        lengths.append(len(episode));wins.append(won)
        paths.append([item[0][:2] for item in episode]+[env.state[:2]])
    return np.array(lengths),np.array(wins),paths

test_env=RaceTrack(build_uturn_course(),seed=909)
test_lengths,test_wins,test_paths=evaluate_greedy(test_env,Q)
print(f'Greedy evaluation: {sum(test_wins)}/{len(test_wins)} successes ({test_wins.mean():.1%})')
print(f'Mean evaluation episode length: {test_lengths.mean():.1f} steps')

In [ ]:
def plot_trajectory(env,path,title='Example learned trajectory'):
    palette={'W':0,'o':1,'-':2,'+':3}
    img=np.vectorize(palette.get)(env.grid)
    fig,ax=plt.subplots(figsize=(8,5));ax.imshow(img,cmap=plt.get_cmap('viridis',4),vmin=0,vmax=3)
    yy,xx=zip(*path)
    ax.plot(xx,yy,'o-',color='white',markersize=4,linewidth=1.6)
    ax.set_title(title);plt.show()

successful=[i for i,w in enumerate(test_wins) if w]
idx=successful[0] if successful else 0
plot_trajectory(test_env,test_paths[idx],f'U-turn greedy policy: {"finish" if test_wins[idx] else "not finished"}')

## 3. Extra challenge: rectangular course

This course needs a longer sequence of coordinated turns. Random exploration is less likely to generate complete successful episodes, and learning a good policy may require **many more episodes** than on the U-turn. This is a computational challenge, not evidence that the algorithm is incorrect.

The same MC code works without modification. To keep a first local run practical, training uses a *modest* budget. For a serious experiment increase `RECT_EPISODES` to 100_000 or more. **No convergence or optimality is guaranteed.**

In [ ]:
rect_env=RaceTrack(build_rect_course(),seed=SEED)
show_course(rect_env,'Extra challenge: rectangular course')

RECT_EPISODES=3000  # increase for experimentation
rect_Q,rect_visits,rect_lengths,rect_wins=mc_control(
    rect_env,episodes=RECT_EPISODES,epsilon=0.15,max_steps=200,
    initial_q=-200.,seed=888)
rect_eval_env=RaceTrack(build_rect_course(),seed=456)
rl, rw, rp=evaluate_greedy(rect_eval_env,rect_Q,n=30)
print(f'Rectangular course greedy evaluation success: {rw.mean():.1%}')
print('Tip: try longer training, alternative epsilon values, and different random seeds.')
plot_trajectory(rect_eval_env,rp[0],'Rectangular course: greedy attempt')

## Key takeaways / self-check questions

1. **Prediction vs. control:** Prediction estimates \(V^\pi\) for a fixed policy; control learns \(Q\) and improves action selection.
2. **Why first visit?** Each unique state (or state-action pair) is updated once per episode using the return after its earliest occurrence.
3. **Why ε-greedy?** To balance exploration of alternatives with use of currently best actions.
4. **Why cap episodes?** Without a cap, a policy might wander indefinitely; episode returns and runtime become problematic.
5. **Why is the harder track harder?** Longer successful trajectories are rare, so informative returns propagate slowly.
6. **Is the policy guaranteed optimal?** No. Finite tabular MC training with limited exploration gives empirical improvement, not a general optimality guarantee.

**Optional experiments:** vary ε, γ, maximum episode length, and training budget; report *held-out* finish rate and episode length. Compare policies across at least three random seeds.

**Source:** Adapted for teaching from the publicly provided Exercise_3 at https://github.com/TejaswiniMedi/reinforcement_learning_hws_2026/tree/main/Exercise_3 . This is a standalone educational reimplementation; its collision handling and policy initialization are explicitly documented above.